<a href="https://colab.research.google.com/github/derekhatherley/L03-tree-stability/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Trees and Model Stability

Trees are notorious for being **unstable**: Small changes in the data can lead to noticeable or large changes in the tree. We're going to explore this phenomenon.

1. Check out the data set `heart_failure` and appropriately clean it.
2. Perform a train-test split for a specific seed (save the seed for reproducibility). Fit a tree and a linear model on the training data and evaluate their performance on the test data. Set aside the predictions these models make.
3. Repeat step 2 for three to five different seeds (save the seeds for reproducibility). How different are the trees that you get? Your linear model coefficients?. Set aside the predictions these models make.
4. Typically, you would see the trees changing what appears to be a non-trivial amount, while the linear model coefficients don't vary nearly as much. Often, the changes appear substantial.
But are they? Instead of focusing on the tree or model coefficients, do three things:
    
    a. Make scatterplots of the predicted values on the test set from question 2 against the predicted values for the alternative models from part 3, separately for your trees and linear models. Do they appear reasonably similar?
    
    b. Compute the correlation between your model in part 2 and your alternative models in part 3, separately for your trees and linear models. Are they highly correlated or not?
    
    c. Run a simple linear regression of the predicted values on the test set from the alternative models on the predicted values from question 2, separately for your trees and linear models. Is the intercept close to zero? Is the slope close to 1? Is the $R^2$ close to 1?

5. Do linear models appear to have similar coefficients and predictions across train/test splits? Do trees?
6. True or false for this dataset, and explain: "Even if the models end up having a substantially different appearance, the predictions they generate are often very similar."

---

## *Phase 1: Solution without AI

Complete all questions in the following cells without generative AI. Then commit and push the
notebook before beginning Phase 2.

**Declaration:** I completed this version without generative AI.

**Student(s):** [To be provided]

**Date:** [To be provided]

In [ ]:
#q1- data exploration and cleaning
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

df = pd.read_csv("data/heart_failure/heart_failure_clinical_records_dataset.csv")
display(df.head())
print(df.shape)
print(df.isna().sum())
display(df.describe())
print(df.duplicated().sum())
#299 by 13, No missing values, no duplicated values

#need to remove time column, since we are interested in if the patient
#died during the follow-up period and follow up ends when a patient dies (leakage)
display(df.groupby('DEATH_EVENT')['time'].median())
df = df.drop(columns = ['time'])
display(df.groupby('DEATH_EVENT').median())

,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
0,75.0,0,582,0,20,1,265000.00,1.9,130,1,0,4,1
1,55.0,0,7861,0,38,0,263358.03,1.1,136,1,0,6,1
2,65.0,0,146,0,20,0,162000.00,1.3,129,1,1,7,1
3,50.0,1,111,0,20,0,210000.00,1.9,137,1,0,7,1
4,65.0,1,160,1,20,0,327000.00,2.7,116,0,0,8,1


(299, 13)
age                         0
anaemia                     0
creatinine_phosphokinase    0
diabetes                    0
ejection_fraction           0
high_blood_pressure         0
platelets                   0
serum_creatinine            0
serum_sodium                0
sex                         0
smoking                     0
time                        0
DEATH_EVENT                 0
dtype: int64


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking,time,DEATH_EVENT
count,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,299.000000,299.00000,299.000000,299.000000,299.00000,299.000000,299.00000
mean,60.833893,0.431438,581.839465,0.418060,38.083612,0.351171,263358.029264,1.39388,136.625418,0.648829,0.32107,130.260870,0.32107
std,11.894809,0.496107,970.287881,0.494067,11.834841,0.478136,97804.236869,1.03451,4.412477,0.478136,0.46767,77.614208,0.46767
min,40.000000,0.000000,23.000000,0.000000,14.000000,0.000000,25100.000000,0.50000,113.000000,0.000000,0.00000,4.000000,0.00000
25%,51.000000,0.000000,116.500000,0.000000,30.000000,0.000000,212500.000000,0.90000,134.000000,0.000000,0.00000,73.000000,0.00000
50%,60.000000,0.000000,250.000000,0.000000,38.000000,0.000000,262000.000000,1.10000,137.000000,1.000000,0.00000,115.000000,0.00000
75%,70.000000,1.000000,582.000000,1.000000,45.000000,1.000000,303500.000000,1.40000,140.000000,1.000000,1.00000,203.000000,1.00000
max,95.000000,1.000000,7861.000000,1.000000,80.000000,1.000000,850000.000000,9.40000,148.000000,1.000000,1.00000,285.000000,1.00000


0


,time
DEATH_EVENT,
0,172.0
1,44.5


,age,anaemia,creatinine_phosphokinase,diabetes,ejection_fraction,high_blood_pressure,platelets,serum_creatinine,serum_sodium,sex,smoking
DEATH_EVENT,,,,,,,,,,,
0,60.0,0.0,245.0,0.0,38.0,0.0,263000.0,1.0,137.0,1.0,0.0
1,65.0,0.0,259.0,0.0,30.0,0.0,258500.0,1.3,135.5,1.0,0.0


In [ ]:
#q2 test train split
from sklearn.model_selection import train_test_split

seed = 100

X = df.drop(columns = ['DEATH_EVENT'])
y = df["DEATH_EVENT"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state= seed)
X_test_q2 = X_test.copy()

print("Train: ",  len(X_train))
print("Test: ", len(X_test))


#tree
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(max_depth = 3, random_state=seed).fit(X_train, y_train)

#set aside
tree_pred = tree.predict(X_test)

#accuracy
print("tree test accuracy: ", np.mean(y_test == tree_pred))

display(pd.crosstab(y_test, tree_pred, rownames=['Actual'], colnames=['Predicted']))

#linear model
from sklearn.linear_model import LinearRegression

linear = LinearRegression().fit(X_train, y_train)

#set aside
linear_pred = linear.predict(X_test)

residuals = y_test - linear_pred
rsq = linear.score(X_test, y_test)
rmse = np.sqrt(np.mean(residuals **2 ))

print("R^2: ", rsq)
print("RMSE: ", rmse)

Train:  239
Test:  60
tree test accuracy:  0.75


Predicted,0,1
Actual,,
0,32,7
1,8,13


R^2:  0.13683194590903713
RMSE:  0.4431373740790705


In [ ]:
#q3-repeat step 2 for 4 more seeds

seeds = [1, 2, 3, 4]

#set aside
tree_preds = np.zeros((len(seeds), len(X_test)))
linear_preds = np.zeros((len(seeds), len(X_test)))
tree_q2_preds = {}
linear_q2_preds = {}

for s in range(len(seeds)):

  X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state= seeds[s])

  tree = DecisionTreeClassifier(max_depth = 3, random_state=seeds[s]).fit(X_train, y_train)
  tree_preds[s]= tree.predict(X_test)

  linear = LinearRegression().fit(X_train, y_train)
  linear_preds[s]= linear.predict(X_test)
  tree_q2_preds[seeds[s]] = tree.predict(X_test_q2)
  linear_q2_preds[seeds[s]] = linear.predict(X_test_q2)

  print("Seed", seeds[s])
  print("tree test accuracy: ", np.mean(y_test == tree_preds[s]))
  print("R^2: ", linear.score(X_test, y_test))
  print("RMSE: ", np.sqrt(np.mean((y_test - linear_preds[s]) **2)))
  display(pd.Series(tree.feature_importances_, index=X.columns))
  display(pd.Series(linear.coef_, index=X.columns))
  print(" ")

Seed 1
tree test accuracy:  0.7666666666666667
R^2:  0.15910299041368992
RMSE:  0.3878493672998376


,0
age,0.222332
anaemia,0.000000
creatinine_phosphokinase,0.000000
diabetes,0.000000
ejection_fraction,0.227727
high_blood_pressure,0.000000
platelets,0.000000
serum_creatinine,0.421100
serum_sodium,0.128841
sex,0.000000


,0
age,8.360179e-03
anaemia,5.424748e-02
creatinine_phosphokinase,5.398486e-05
diabetes,5.810835e-02
ejection_fraction,-9.974917e-03
high_blood_pressure,3.702920e-02
platelets,-2.597075e-07
serum_creatinine,1.145516e-01
serum_sodium,-1.216894e-02
sex,-5.978601e-02


 
Seed 2
tree test accuracy:  0.75
R^2:  0.1922578914239843
RMSE:  0.40498953394195103


,0
age,0.225618
anaemia,0.000000
creatinine_phosphokinase,0.000000
diabetes,0.000000
ejection_fraction,0.310376
high_blood_pressure,0.000000
platelets,0.000000
serum_creatinine,0.379252
serum_sodium,0.031723
sex,0.053030


,0
age,8.916852e-03
anaemia,6.246412e-02
creatinine_phosphokinase,6.624411e-05
diabetes,-1.162067e-02
ejection_fraction,-9.656941e-03
high_blood_pressure,4.477614e-02
platelets,-1.489501e-07
serum_creatinine,1.110636e-01
serum_sodium,-9.860709e-03
sex,-8.048512e-02


... (2 outputs left out)

,0
age,9.791005e-03
anaemia,4.024142e-03
creatinine_phosphokinase,7.695639e-05
diabetes,-6.584872e-03
ejection_fraction,-9.929368e-03
high_blood_pressure,6.820250e-02
platelets,-1.929315e-08
serum_creatinine,1.093293e-01
serum_sodium,-1.280425e-02
sex,-5.997680e-02


 
Seed 4
tree test accuracy:  0.7166666666666667
R^2:  0.1034775530013341
RMSE:  0.43390058062846587


,0
age,0.199907
anaemia,0.000000
creatinine_phosphokinase,0.000000
diabetes,0.000000
ejection_fraction,0.214646
high_blood_pressure,0.000000
platelets,0.000000
serum_creatinine,0.444471
serum_sodium,0.140975
sex,0.000000


,0
age,1.063605e-02
anaemia,5.560508e-02
creatinine_phosphokinase,3.925957e-05
diabetes,6.678660e-02
ejection_fraction,-1.004854e-02
high_blood_pressure,6.537906e-02
platelets,-6.770158e-08
serum_creatinine,1.034946e-01
serum_sodium,-1.155249e-02
sex,-2.047693e-02


**Q3** 

How different are the trees?

For each of the four seeds, serum creatinine has the highest feature importance, followed by ejection fraction and age. Between seeds, these importance values fluctuate, and the less important features vary slightly. Test accuracy ranges from .72 to .77.

Your linear model coefficients?

The coefficients for age and ejection fraction fluctuate only slightly (the coefficient for age is about .01 and the coefficient for ejection fraction is about −.01). Some other coefficients vary more (for example, the coefficient for diabetes ranges from about −.0116 to .0668 across the different seeds).

In [ ]:
#Q4: 

fig, axes = plt.subplots(2, 4, figsize=(13, 6), constrained_layout=True)
results = []

for col, s in enumerate(seeds):
    comparisons = [
        ("Tree", tree_pred, tree_q2_preds[s]),
        ("Linear", linear_pred, linear_q2_preds[s])
    ]

    for row, (name, baseline, alternative) in enumerate(comparisons):
        ax = axes[row, col]

        if name == "Tree":
            pairs, counts = np.unique(
                np.column_stack([baseline, alternative]),
                axis=0, return_counts=True
            )
            ax.scatter(pairs[:, 0], pairs[:, 1], s=12 * counts, alpha=0.65)
            ax.set_xticks([0, 1])
            ax.set_yticks([0, 1])
            limits = (-0.2, 1.2)
        else:
            ax.scatter(baseline, alternative, s=18, alpha=0.65)
            limits = (min(baseline.min(), alternative.min()) - 0.05,
                      max(baseline.max(), alternative.max()) + 0.05)

        ax.plot(limits, limits, "--", color="gray", linewidth=1)
        ax.set(xlim=limits, ylim=limits, aspect="equal",
               title=f"{name}, seed {s}", xlabel="Seed 100 prediction")
        if col == 0:
            ax.set_ylabel("Alternative prediction")

        fit = LinearRegression().fit(baseline.reshape(-1, 1), alternative)
        results.append([name, s, np.corrcoef(baseline, alternative)[0, 1],
                        fit.intercept_, fit.coef_[0],
                        fit.score(baseline.reshape(-1, 1), alternative)])

plt.show()
display(pd.DataFrame(results, columns=[
    "Model", "Seed", "Correlation", "Intercept", "Slope", "R²"
]).sort_values(["Model", "Seed"]).round(3))

**4a.** The linear-model predictions appear similar across all four splits. The points cluster around the diagonal, so a patient who received a high prediction from the seed-100 model generally received a similarly high prediction from each alternative model. For the trees, the two points on the diagonal represent agreement on the predicted class. Most patients fall on the diagonal, but depending on the seed, 8 to 12 of the 60 patients changed predicted class.

**4b.** The linear-model predictions were highly correlated with the seed-100 predictions across all four splits, with correlations ranging from 0.966 to 0.969. The tree correlations varied more. They were 0.572, 0.572, 0.709, and 0.577 for seeds 1 through 4, respectively. The seed-3 tree agreed most closely with the original tree, assigning the same class to 52 of the 60 patients.

**4c.** For the linear models, the intercepts were close to 0 (0.015–0.051), the slopes were close to 1 (0.926–1.015), and the R² values were also close to 1 (0.934–0.939). For the trees, the intercepts ranged from 0.075 to 0.200, so some were close to 0 and others were noticeably higher. All four slopes fell well below 1 (0.525–0.725), and none of the R² values were close to 1 (0.327–0.503). These regressions measure how closely the alternative models reproduce the original model’s predictions for the same patients.

**Q5** 

The linear models produced similar predictions across splits even though some coefficients changed. For example, the diabetes coefficient changed sign, but the predictions for the same 60 patients still tracked closely across all four splits.

The trees were stable in which features they ranked highest, but less stable in the predictions they made for individual patients. Their test accuracies ranged from about 72% to 77%, which looks fairly consistent across splits. However, when I compared the trees on the same 60 patients, 13% to 20% received a different predicted class depending on the seed.

## *Phase 2: AI-assisted Revision
Keep Phase 1 frozen.

- **Phase 1 commit SHA ID:** [To be provided]
- **AI tool and model used:** [To be provided]

### *Prompts used

1. [Paste prompt]
2. [Paste follow-up prompt]

### *AI-proposed changes


[Paste the AI's concise numbered list verbatim.]

1. ...
2. ...

### *Evaluation of AI suggestions


| Change | Decision | Reason and verification |
|---|---|---|
| 1 | Accept / modify / reject | ... |
| 2 | Accept / modify / reject | ... |

### *AI-assisted solution in full
After the evaluation of AI suggestions, incorporate the accepted revisions into your Phase 1 solution and provide the final Phase 2 solution in full in the following cells.

In [ ]:
# Your Phase 2 solution to the problem goes here.


### *Reflection on AI assistance
In your own words without generative AI.

[In 150–300 words, describe the main improvements, AI mistakes or limitations, how the final solution was verified, and any remaining concerns.]